[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [8]:
from re import X
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):

      self.d_model = d_model
      self.num_heads = num_heads
      self.num_kv_heads = num_kv_heads

      self.d_k = d_model // num_heads

      self.W_q = nn.Linear(d_model, d_model)
      self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
      self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
      self.W_o = nn.Linear(d_model, d_model)


        # pass  # Initialize projections

    def forward(self, x):
      B, S, D = x.shape

      # (B, S, D)
      q = self.W_q(x)
      k = self.W_k(x)
      v = self.W_v(x)
      print(f"projection shapes: {q.shape}\n\n")

      # Separate into heads
      q = q.view(B, S, self.num_heads, self.d_k)
      k = k.view(B, S, self.num_kv_heads, self.d_k)
      v = v.view(B, S, self.num_kv_heads, self.d_k)

      print(f"after head separation before repeat shapes: {q.shape}")
      print(f"after head separation before repeat shapes: {k.shape}")
      print(f"after head separation before repeat shapes: {v.shape}\n\n")


      # torch.repeat_interleave for repeating kv heads to match num_q heads

      repeat_factor = self.num_heads // self.num_kv_heads
      # this below will change shapes to (B, S, num_heads, d_k)

      if self.num_kv_heads != self.num_heads:  # works like regular MHA otherwise
        k = torch.repeat_interleave(k, repeat_factor, dim=2)
        v = torch.repeat_interleave(v, repeat_factor, dim=2)

      print(f"after repeat shapes: {q.shape}")
      print(f"after repeat shapes: {k.shape}")
      print(f"after repeat shapes: {v.shape}")


      # Transpose for attention later on, at this point, same as regular MHA

      q = q.transpose(1,2) # (B, H, S, d_k)
      k = k.transpose(1,2)
      v = v.transpose(1,2) # (B, H_kv, S_kv, d_k) for both k and v

      k = k.transpose(-1, -2) # (B, H_kv, d_k, S_kv)

      scores = torch.matmul(q, k) / math.sqrt(self.d_k) # (B, H, S, S_kv)
      weights = torch.softmax(scores, dim=-1)
      attn_output = torch.matmul(weights, v) # (B, H_kv, S, d_k)

      attn_output = attn_output.transpose(1,2).contiguous()
      x = attn_output.view(B, S, D)

      return self.W_o(x)






In [9]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
projection shapes: torch.Size([2, 6, 32])


after head separation before repeat shapes: torch.Size([2, 6, 8, 4])
after head separation before repeat shapes: torch.Size([2, 6, 2, 4])
after head separation before repeat shapes: torch.Size([2, 6, 2, 4])


after repeat shapes: torch.Size([2, 6, 8, 4])
after repeat shapes: torch.Size([2, 6, 8, 4])
after repeat shapes: torch.Size([2, 6, 8, 4])
Output shape: torch.Size([2, 6, 32])


In [10]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
projection shapes: torch.Size([2, 6, 32])


after head separation before repeat shapes: torch.Size([2, 6, 8, 4])
after head separation before repeat shapes: torch.Size([2, 6, 2, 4])
after head separation before repeat shapes: torch.Size([2, 6, 2, 4])


after repeat shapes: torch.Size([2, 6, 8, 4])
after repeat shapes: torch.Size([2, 6, 8, 4])
after repeat shapes: torch.Size([2, 6, 8, 4])
  ✅ [1/5] Output shape (2.9ms)
  ✅ [2/5] nn.Linear with correct shapes (0.9ms)
projection shapes: torch.Size([1, 4, 16])


after head separation before repeat shapes: torch.Size([1, 4, 4, 4])
after head separation before repeat shapes: torch.Size([1, 4, 4, 4])
after head separation before repeat shapes: torch.Size([1, 4, 4, 4])


after repeat shapes: torch.Size([1, 4, 4, 4])
after repeat shapes: torch.Size([1, 4, 4, 4])
after repeat shapes: torch.Size([1, 4, 4, 4])
  ✅ [3/5] Degenerates to MHA when kv_heads ==